In [3]:
"""
Full-article resolution for entities that step0_abbreviation_expansion.py
could NOT resolve, category 1: no definition anywhere in the paper's
extracted `corresponding_sentence` values, only in the article as a
whole (often the abstract or methods section).

Batches by PAPER, not by code: one LLM call per paper, with that
paper's full assembled text plus the list of every unresolved code in
it, rather than one call per code (which would resend the whole
article once per code, far more expensive for no benefit).

Filters out genotype/sample IDs before sending anything to the LLM,
those don't have a "real name" to resolve to (SP-10 IS the identity,
not a code standing in for something else), asking an LLM to find one
would waste a call and could produce a fabricated answer.

Requires OPENAI_API_KEY. Uses a cheap/fast model by default (swap
LLM_MODEL for your Luna/Gemini-cheap deployment's model string).
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PART-1-PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
CHUNKS_DOI_COL = "doi"
CHUNKS_TEXT_COL = "chunk_text"
CHUNKS_INDEX_COL = "chunk_index"

RESIDUAL_XLSX = "abbreviation_expansion_review.xlsx"  # output of step0, "still_unresolved" sheet

LLM_MODEL = "gpt-4o-mini"  # swap for your Luna / Gemini-cheap model string
MAX_ARTICLE_CHARS = 60000  # safety cap; corpus average is ~38k, max seen ~89k

OUTPUT_XLSX = "full_article_resolution_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# STEP 1: LOAD THE RESIDUAL LIST AND FILTER OUT ID-STYLE CODES
# ---------------------------------------------------------------
# same code-token logic as tier2, used here only to RECOGNIZE and
# EXCLUDE genotype/sample IDs, not to resolve anything
_CODE_TOKEN = r"[A-Za-z0-9]+(?:[\-\u2010\u2011\u2012\u2013\u2014\u2212][A-Za-z0-9]+)*"
_ID_STYLE_PATTERN = re.compile(
    r"\b(?:sample|genotype|genotypes|variant|cultivar)\s+" + _CODE_TOKEN + r"\b",
    re.IGNORECASE,
)
_HYPHENATED_ID_PATTERN = re.compile(r"^[A-Za-z]{1,4}[\-\u2013\u2014]\d+$")  # e.g. SP-10, N16-10044


def is_id_style_code(entity_text, token):
    """
    True if `token` looks like a sample/genotype ID (has a real
    identity of its own, nothing to resolve) rather than an
    abbreviation standing in for something else.
    """
    if _HYPHENATED_ID_PATTERN.match(token):
        return True
    if _ID_STYLE_PATTERN.search(entity_text):
        return True
    return False


def token_is_id_prefix_anywhere_in_paper(token, doi, article_text_by_doi):
    """
    Paper-wide check: does this bare token appear followed by a hyphen
    and a number ANYWHERE in this paper's full text (e.g. "SP-10",
    "SP-17"), even if the specific entity where the token was extracted
    didn't happen to have "genotype"/"sample" sitting right next to it?

    This catches cases the per-entity check misses: a paper can
    define "SPI powders (SP-1-SP-20)" in one sentence and just say
    "SP-7" elsewhere without the word "genotype" nearby, so the
    per-entity check alone would wrongly treat bare "SP" as a real
    abbreviation needing resolution, when it's actually a genotype
    prefix throughout the paper. Confirmed necessary on this corpus:
    "SP" in DOI 10-1016_j-fochms-2026-100388 was incorrectly resolved
    by the LLM as "Soy Protein" (marked high confidence) when every
    real occurrence in the article is either "SPI" (soy protein
    isolate) or the genotype prefix "SP-1" through "SP-20", never a
    standalone "SP" abbreviation.
    """
    text = article_text_by_doi.get(doi, "")
    return bool(re.search(rf"\b{re.escape(token)}[\-\u2013\u2014]\d", text))


residual_df = pd.read_excel(RESIDUAL_XLSX, sheet_name="still_unresolved")
print(f"Loaded {len(residual_df)} unresolved (doi, entity, token) rows")

residual_df["is_id_style"] = residual_df.apply(
    lambda r: is_id_style_code(str(r["entity"]), str(r["undefined_token"])), axis=1
)
print(f"Filtered out as genotype/sample IDs (per-entity check): {residual_df['is_id_style'].sum()}")

# ---------------------------------------------------------------
# STEP 2: LOAD AND ASSEMBLE FULL ARTICLE TEXT PER DOI
# (loaded here, before the paper-wide ID check, since that check needs it)
# ---------------------------------------------------------------
chunks_df = pd.read_parquet(CHUNKS_PATH)

article_text_by_doi = (
    chunks_df.sort_values(CHUNKS_INDEX_COL)
    .groupby(CHUNKS_DOI_COL)[CHUNKS_TEXT_COL]
    .apply(lambda s: "\n\n".join(s))
    .to_dict()
)

# paper-wide second pass: catches tokens the per-entity check missed,
# like "SP" in a paper that also has "SP-1" through "SP-20" elsewhere
still_not_id = residual_df[~residual_df["is_id_style"]].copy()
still_not_id["is_id_prefix_paperwide"] = still_not_id.apply(
    lambda r: token_is_id_prefix_anywhere_in_paper(str(r["undefined_token"]), r["doi"], article_text_by_doi),
    axis=1,
)
print(f"Additionally filtered by paper-wide check: {still_not_id['is_id_prefix_paperwide'].sum()}")

to_resolve = still_not_id[~still_not_id["is_id_prefix_paperwide"]].copy()
skipped_ids = pd.concat([
    residual_df[residual_df["is_id_style"]],
    still_not_id[still_not_id["is_id_prefix_paperwide"]].drop(columns="is_id_prefix_paperwide"),
])

print(f"Total filtered out as genotype/sample IDs: {len(skipped_ids)}")
print(f"Remaining, sent to full-article resolution: {len(to_resolve)}")

# ---------------------------------------------------------------
# STEP 3: ONE LLM CALL PER PAPER
# ---------------------------------------------------------------
# GENERAL safeguard (not shape-specific): the LLM must quote the exact
# sentence it found the definition in. The script then independently
# verifies that quote actually appears in the article before accepting
# the answer. This catches ANY wrong resolution, regardless of what
# caused it, not just the one failure shape (a code that's really a
# genotype ID) caught by the earlier filters. Those filters stay in
# place too, as a cheap first pass, but they are no longer the last
# line of defense, verification is.
RESOLUTION_PROMPT = """This is the full text of a plant protein research paper.

Find the real, full meaning of each of these codes/abbreviations as used
IN THIS PAPER (they may be defined in the abstract, methods, or anywhere
else in the text, often as "full phrase (CODE)"):

{codes}

For each code:
- Only report a full_form if you can point to an EXACT sentence in the
  article text below that defines or clearly supports it. Copy that
  sentence into "supporting_sentence" VERBATIM, character-for-character
  from the article, do not paraphrase or reconstruct it from memory.
- If you cannot find an exact sentence that supports a definition,
  return full_form: null and supporting_sentence: null. Do not guess
  or infer a plausible-sounding answer with no exact quote behind it.

Article text:
{article}

Respond ONLY with JSON, no markdown fences, a list of objects:
[{{"code": "...", "full_form": "..." or null, "supporting_sentence": "..." or null}}]
"""


def normalize_for_match(text):
    """Collapse whitespace/soft-hyphen artifacts so verbatim quotes
    still match even if line breaks or soft hyphens differ slightly."""
    text = text.replace("\xad", "").replace("­", "")  # soft hyphen variants
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def verify_quote_in_article(quoted_sentence, article_text):
    """
    Independent check: does the LLM's quoted supporting_sentence
    actually appear in the article? This is what makes the safeguard
    general rather than specific, it doesn't matter WHY an answer
    might be wrong, an unverifiable quote is rejected regardless.
    """
    if not quoted_sentence:
        return False
    return normalize_for_match(quoted_sentence) in normalize_for_match(article_text)


def looks_like_data_table_row(sentence):
    """
    A quote can genuinely exist in the article (pass verify_quote_in_article)
    while STILL not being a definition, if the LLM quoted a data table
    row that merely happens to contain the code, rather than a sentence
    that defines it. Confirmed real on this corpus: "AOT | EPI 80 |
    17.33 +/- 0.67 h | 7.14 +/- 0.01 g | ..." verified as an existing
    quote, but it's tabulated data, not a definition.

    Signature checked: a pipe character (table cell separator), or the
    "value +/- value LETTER" pattern (mean +/- SD with a Duncan's
    multiple range test grouping letter), which is unambiguous in a
    way generic short-word counting isn't, an earlier version counted
    any short lowercase token and falsely flagged ordinary prose
    containing words like "is" or "of".
    """
    if "|" in sentence:
        return True
    stat_pattern = re.findall(r"[\u00b1+/-]{1,2}\s*\d+\.?\d*\s+[a-z]{1,3}\b", sentence.lower())
    if len(stat_pattern) >= 2:
        return True
    return False


def full_form_looks_like_real_name(full_form):
    """
    A real definition contains at least one word (3+ letters) that
    is NOT all-uppercase, e.g. "corn germ meal" or "Alcohol
    dehydrogenase". A bare code-like answer such as "E86 HV" is only
    all-caps/alphanumeric tokens, no real descriptive word at all,
    confirmed as a real failure case on this corpus: "HV" resolved to
    "E86 HV", which is just another sample code, not an expansion.
    """
    words = re.findall(r"[A-Za-z]{3,}", str(full_form))
    return any(not w.isupper() for w in words)


def full_form_adds_no_information(full_form, code):
    """
    Reject answers where "full_form" is really just the code again,
    optionally with a number/other code stuck on, e.g. code "EPI"
    resolving to "EPI 80", or code "HV" resolving to "E86 HV". Neither
    is a real expansion, the abbreviation was never actually resolved
    to a descriptive name.
    """
    if not full_form:
        return True
    ff = str(full_form).strip()
    code_clean = str(code).strip()
    if ff.upper() == code_clean.upper():
        return True
    remainder = ff[len(code_clean):] if ff.upper().startswith(code_clean.upper()) else None
    if remainder is not None and re.fullmatch(r"[\s\d.,%°/–\-]*", remainder):
        return True
    if not full_form_looks_like_real_name(ff):
        return True
    return False


def resolve_paper_codes(article_text, codes):
    if len(article_text) > MAX_ARTICLE_CHARS:
        article_text = article_text[:MAX_ARTICLE_CHARS]
    prompt = RESOLUTION_PROMPT.format(codes="\n".join(f"- {c}" for c in codes), article=article_text)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    time.sleep(0.2)
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = [{"code": c, "full_form": None, "supporting_sentence": None} for c in codes]

    # verification pass: THREE independent checks, all three must pass
    # for an answer to be trusted. Quote-existence alone (the original
    # check) isn't enough, a quote can be real and still not be a
    # definition (a data table row), so this also checks that the quote
    # reads like prose rather than tabulated data, and that the answer
    # actually expands the code rather than just restating it.
    for r in parsed:
        code = r.get("code")
        full_form = r.get("full_form")
        quote = r.get("supporting_sentence")

        quote_exists = verify_quote_in_article(quote, article_text)
        is_table_row = looks_like_data_table_row(quote) if quote else False
        adds_nothing = full_form_adds_no_information(full_form, code)

        r["quote_verified"] = quote_exists
        r["looks_like_table_row"] = is_table_row
        r["full_form_adds_information"] = not adds_nothing

        if quote_exists and not is_table_row and not adds_nothing:
            r["status"] = "verified"
        else:
            r["status"] = "unverified_needs_review"
    return parsed


results = []
papers_processed = 0
for doi, group in to_resolve.groupby("doi"):
    codes = sorted(group["undefined_token"].unique())
    article_text = article_text_by_doi.get(doi)
    if not article_text:
        for c in codes:
            results.append({"doi": doi, "code": c, "full_form": None,
                             "supporting_sentence": None, "quote_verified": False,
                             "status": "no_article_text_available"})
        continue

    resolutions = resolve_paper_codes(article_text, codes)
    for r in resolutions:
        results.append({
            "doi": doi,
            "code": r.get("code"),
            "full_form": r.get("full_form"),
            "supporting_sentence": r.get("supporting_sentence"),
            "quote_verified": r.get("quote_verified"),
            "status": r.get("status"),
        })
    papers_processed += 1

resolution_df = pd.DataFrame(results)
n_verified = (resolution_df["status"] == "verified").sum()
n_unverified = (resolution_df["status"] == "unverified_needs_review").sum()
print(f"\nPapers processed: {papers_processed}")
print(f"Verified (quote confirmed present in article): {n_verified} of {len(resolution_df)}")
print(f"Unverified (LLM's quote could not be confirmed, NOT auto-applied): {n_unverified}")

# ---------------------------------------------------------------
# SAVE FOR REVIEW
# ---------------------------------------------------------------
verified_df = resolution_df[resolution_df["status"] == "verified"]
unverified_df = resolution_df[resolution_df["status"] != "verified"]

with pd.ExcelWriter(OUTPUT_XLSX) as writer:
    verified_df.to_excel(writer, sheet_name="verified_resolutions", index=False)
    unverified_df.to_excel(writer, sheet_name="unverified_needs_review", index=False)
    skipped_ids.to_excel(writer, sheet_name="skipped_id_style_codes", index=False)

print(f"\nSaved to {OUTPUT_XLSX}")
print("verified_resolutions: quote independently confirmed present in the article, safe to apply.")
print("unverified_needs_review: LLM gave an answer but its quote couldn't be confirmed, do NOT")
print("  apply these without a manual check, this is the general safeguard, not a specific one,")
print("  it catches any wrong answer regardless of what caused it.")
print("skipped_id_style_codes: genotype/sample IDs correctly excluded, no name to find for these.")

Loaded 299 unresolved (doi, entity, token) rows
Filtered out as genotype/sample IDs (per-entity check): 68
Additionally filtered by paper-wide check: 24
Total filtered out as genotype/sample IDs: 92
Remaining, sent to full-article resolution: 207

Papers processed: 74
Verified (quote confirmed present in article): 82 of 125
Unverified (LLM's quote could not be confirmed, NOT auto-applied): 43

Saved to full_article_resolution_review.xlsx
verified_resolutions: quote independently confirmed present in the article, safe to apply.
unverified_needs_review: LLM gave an answer but its quote couldn't be confirmed, do NOT
  apply these without a manual check, this is the general safeguard, not a specific one,
  it catches any wrong answer regardless of what caused it.
skipped_id_style_codes: genotype/sample IDs correctly excluded, no name to find for these.
